In [ ]:
!python train_t5_relation.py

In [ ]:
%pip install shap

In [ ]:
import shap
import pandas as pd
import torch
from transformers import pipeline, RobertaTokenizer, RobertaForSequenceClassification

try:
    get_ipython
    shap.initjs()
except NameError:
    print("not Notebook environment")

# 1. mapping
train_raw = pd.read_csv('train_t5.csv')
unique_labels = sorted(train_raw['target_text'].dropna().unique().tolist())
id2label = {i: label for i, label in enumerate(unique_labels)}
label2id = {label: i for i, label in enumerate(unique_labels)}

# 2. load the modle
model_path = 'my_roberta_model'
tokenizer = RobertaTokenizer.from_pretrained(model_path)
model = RobertaForSequenceClassification.from_pretrained(model_path)
model.config.id2label = id2label
model.config.label2id = label2id

# 3. pipeline
device_id = 0 if torch.cuda.is_available() else -1
pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device_id,
    top_k=None
)

# SHAP explainer
explainer = shap.Explainer(pipe)

# 4. test the example
test_df = pd.read_csv('test_t5.csv')
test_df['clean_text'] = test_df['input_text'].str.replace('compound: ', '', regex=False)
test_df[['w1', 'w2']] = test_df['clean_text'].str.split(' ', n=1, expand=True)
sample_texts = (test_df['w1'] + " " + test_df['w2']).dropna().head(3).tolist()
print(f"样本: {sample_texts}")

# 5. SHAP analysis
shap_values = explainer(sample_texts)

# 6. Visualization
try:
    shap.plots.text(shap_values)
except Exception:
    import matplotlib.pyplot as plt
    shap.plots.text(shap_values, show=False)
    plt.savefig("shap_text_plot.png")
    print("save to shap_text_plot.png")
